In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# ---------------------------------------------------------
# 1. Load and Prepare the Dataset (HaluEval)
# ---------------------------------------------------------
print("Loading HaluEval dataset...")
# We'll use the QA split of HaluEval for this example
dataset = load_dataset("pminervini/HaluEval", "qa")
df = pd.DataFrame(dataset['data'])

# HaluEval has a 'knowledge', 'question', 'right_answer', and 'hallucinated_answer'
# We need to restructure this into a binary classification format: 1 (Hallucination), 0 (Factual)

# Create factual samples (Label 0)
df_factual = df[['question', 'right_answer']].copy()
df_factual.columns = ['prompt', 'response']
df_factual['label'] = 0

# Create hallucinated samples (Label 1)
df_hallucinated = df[['question', 'hallucinated_answer']].copy()
df_hallucinated.columns = ['prompt', 'response']
df_hallucinated['label'] = 1

# Combine and shuffle
df_combined = pd.concat([df_factual, df_hallucinated]).sample(frac=1, random_state=42).reset_index(drop=True)

# To give the classifier context, we combine the prompt and the response
df_combined['text_to_analyze'] = df_combined['prompt'] + " [SEP] " + df_combined['response']

# For rapid testing, let's take a subset (remove this line for the full run)
df_subset = df_combined.head(10000)

X_train, X_test, y_train, y_test = train_test_split(
    df_subset['text_to_analyze'], df_subset['label'], test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# 2. Feature Extraction: TF-IDF & N-Grams
# ---------------------------------------------------------
print("Extracting TF-IDF and N-gram features...")
# Using unigrams and bigrams, ignoring terms that appear in >90% of docs or <5 docs
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.90, min_df=5, max_features=10000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ---------------------------------------------------------
# 3. Model Training: SVM & XGBoost
# ---------------------------------------------------------
print("Training Support Vector Machine (SVM)...")
# Using probability=True so we can calculate AUROC and AUPRC later
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train_tfidf, y_train)

print("Training XGBoost...")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_tfidf, y_train)

# ---------------------------------------------------------
# 4. Evaluation (Precision, Recall, F1, AUROC, AUPRC)
# ---------------------------------------------------------
def evaluate_model(model, name, X_test_features, y_test_true):
    print(f"\n--- {name} Evaluation ---")

    # Get standard predictions
    y_pred = model.predict(X_test_features)
    # Get probability scores for the positive class (Hallucination)
    y_prob = model.predict_proba(X_test_features)[:, 1]

    print(classification_report(y_test_true, y_pred, target_names=['Factual (0)', 'Hallucination (1)']))

    # Calculate continuous metrics as per your proposal
    auroc = roc_auc_score(y_test_true, y_prob)
    auprc = average_precision_score(y_test_true, y_prob)

    print(f"AUROC: {auroc:.4f}")
    print(f"AUPRC: {auprc:.4f}")

evaluate_model(svm_model, "SVM", X_test_tfidf, y_test)
evaluate_model(xgb_model, "XGBoost", X_test_tfidf, y_test)

Loading HaluEval dataset...
Extracting TF-IDF and N-gram features...
Training Support Vector Machine (SVM)...
Training XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:48:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- SVM Evaluation ---
                   precision    recall  f1-score   support

      Factual (0)       0.71      0.74      0.72       984
Hallucination (1)       0.74      0.71      0.72      1016

         accuracy                           0.72      2000
        macro avg       0.72      0.72      0.72      2000
     weighted avg       0.72      0.72      0.72      2000

AUROC: 0.7867
AUPRC: 0.8013

--- XGBoost Evaluation ---
                   precision    recall  f1-score   support

      Factual (0)       0.81      0.89      0.85       984
Hallucination (1)       0.88      0.80      0.84      1016

         accuracy                           0.84      2000
        macro avg       0.84      0.84      0.84      2000
     weighted avg       0.85      0.84      0.84      2000

AUROC: 0.9046
AUPRC: 0.9233


In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# ---------------------------------------------------------
# 1. Load and Prepare the Dataset (HaluEval)
# ---------------------------------------------------------
print("Loading HaluEval dataset...")
# We'll use the QA split of HaluEval for this example
dataset = load_dataset("pminervini/HaluEval", "dialogue")
df = pd.DataFrame(dataset['data'])

# HaluEval has a 'knowledge', 'dialogue history', 'right_response', and 'hallucinated_response'
# We need to restructure this into a binary classification format: 1 (Hallucination), 0 (Factual)

# Create factual samples (Label 0)
df_factual = df[['dialogue_history', 'right_response']].copy()
df_factual.columns = ['prompt', 'response']
df_factual['label'] = 0

# Create hallucinated samples (Label 1)
df_hallucinated = df[['dialogue_history', 'hallucinated_response']].copy()
df_hallucinated.columns = ['prompt', 'response']
df_hallucinated['label'] = 1

# Combine and shuffle
df_combined = pd.concat([df_factual, df_hallucinated]).sample(frac=1, random_state=42).reset_index(drop=True)

# To give the classifier context, we combine the prompt and the response
df_combined['text_to_analyze'] = df_combined['prompt'] + " [SEP] " + df_combined['response']

# For rapid testing, let's take a subset (remove this line for the full run)
df_subset = df_combined.head(10000)

X_train, X_test, y_train, y_test = train_test_split(
    df_subset['text_to_analyze'], df_subset['label'], test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# 2. Feature Extraction: TF-IDF & N-Grams
# ---------------------------------------------------------
print("Extracting TF-IDF and N-gram features...")
# Using unigrams and bigrams, ignoring terms that appear in >90% of docs or <5 docs
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.90, min_df=5, max_features=10000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ---------------------------------------------------------
# 3. Model Training: SVM & XGBoost
# ---------------------------------------------------------
print("Training Support Vector Machine (SVM)...")
# Using probability=True so we can calculate AUROC and AUPRC later
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train_tfidf, y_train)

print("Training XGBoost...")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_tfidf, y_train)

# ---------------------------------------------------------
# 4. Evaluation (Precision, Recall, F1, AUROC, AUPRC)
# ---------------------------------------------------------
def evaluate_model(model, name, X_test_features, y_test_true):
    print(f"\n--- {name} Evaluation ---")

    # Get standard predictions
    y_pred = model.predict(X_test_features)
    # Get probability scores for the positive class (Hallucination)
    y_prob = model.predict_proba(X_test_features)[:, 1]

    print(classification_report(y_test_true, y_pred, target_names=['Factual (0)', 'Hallucination (1)']))

    # Calculate continuous metrics as per your proposal
    auroc = roc_auc_score(y_test_true, y_prob)
    auprc = average_precision_score(y_test_true, y_prob)

    print(f"AUROC: {auroc:.4f}")
    print(f"AUPRC: {auprc:.4f}")

evaluate_model(svm_model, "SVM", X_test_tfidf, y_test)
evaluate_model(xgb_model, "XGBoost", X_test_tfidf, y_test)

Loading HaluEval dataset...


dialogue/data-00000-of-00001.parquet:   0%|          | 0.00/3.45M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Extracting TF-IDF and N-gram features...
Training Support Vector Machine (SVM)...
Training XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:59:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- SVM Evaluation ---
                   precision    recall  f1-score   support

      Factual (0)       0.61      0.72      0.66       984
Hallucination (1)       0.67      0.56      0.61      1016

         accuracy                           0.64      2000
        macro avg       0.64      0.64      0.63      2000
     weighted avg       0.64      0.64      0.63      2000

AUROC: 0.6909
AUPRC: 0.7274

--- XGBoost Evaluation ---
                   precision    recall  f1-score   support

      Factual (0)       0.63      0.75      0.69       984
Hallucination (1)       0.70      0.58      0.64      1016

         accuracy                           0.66      2000
        macro avg       0.67      0.66      0.66      2000
     weighted avg       0.67      0.66      0.66      2000

AUROC: 0.7186
AUPRC: 0.7626


In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# ---------------------------------------------------------
# 1. Load and Prepare the Dataset (HaluEval)
# ---------------------------------------------------------
print("Loading HaluEval dataset...")
# We'll use the QA split of HaluEval for this example
dataset = load_dataset("pminervini/HaluEval", "summarization")
df = pd.DataFrame(dataset['data'])

# HaluEval has a 'knowledge', 'question', 'right_answer', and 'hallucinated_answer'
# We need to restructure this into a binary classification format: 1 (Hallucination), 0 (Factual)

# Create factual samples (Label 0)
df_factual = df[['document', 'right_summary']].copy()
df_factual.columns = ['prompt', 'response']
df_factual['label'] = 0

# Create hallucinated samples (Label 1)
df_hallucinated = df[['document', 'hallucinated_summary']].copy()
df_hallucinated.columns = ['prompt', 'response']
df_hallucinated['label'] = 1

# Combine and shuffle
df_combined = pd.concat([df_factual, df_hallucinated]).sample(frac=1, random_state=42).reset_index(drop=True)

# To give the classifier context, we combine the prompt and the response
df_combined['text_to_analyze'] = df_combined['prompt'] + " [SEP] " + df_combined['response']

# For rapid testing, let's take a subset (remove this line for the full run)
df_subset = df_combined.head(10000)

X_train, X_test, y_train, y_test = train_test_split(
    df_subset['text_to_analyze'], df_subset['label'], test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# 2. Feature Extraction: TF-IDF & N-Grams
# ---------------------------------------------------------
print("Extracting TF-IDF and N-gram features...")
# Using unigrams and bigrams, ignoring terms that appear in >90% of docs or <5 docs
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.90, min_df=5, max_features=10000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ---------------------------------------------------------
# 3. Model Training: SVM & XGBoost
# ---------------------------------------------------------
print("Training Support Vector Machine (SVM)...")
# Using probability=True so we can calculate AUROC and AUPRC later
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train_tfidf, y_train)

print("Training XGBoost...")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_tfidf, y_train)

# ---------------------------------------------------------
# 4. Evaluation (Precision, Recall, F1, AUROC, AUPRC)
# ---------------------------------------------------------
def evaluate_model(model, name, X_test_features, y_test_true):
    print(f"\n--- {name} Evaluation ---")

    # Get standard predictions
    y_pred = model.predict(X_test_features)
    # Get probability scores for the positive class (Hallucination)
    y_prob = model.predict_proba(X_test_features)[:, 1]

    print(classification_report(y_test_true, y_pred, target_names=['Factual (0)', 'Hallucination (1)']))

    # Calculate continuous metrics as per your proposal
    auroc = roc_auc_score(y_test_true, y_prob)
    auprc = average_precision_score(y_test_true, y_prob)

    print(f"AUROC: {auroc:.4f}")
    print(f"AUPRC: {auprc:.4f}")

evaluate_model(svm_model, "SVM", X_test_tfidf, y_test)
evaluate_model(xgb_model, "XGBoost", X_test_tfidf, y_test)

Loading HaluEval dataset...


summarization/data-00000-of-00001.parque(…):   0%|          | 0.00/28.0M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Extracting TF-IDF and N-gram features...
Training Support Vector Machine (SVM)...
Training XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [03:32:38] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- SVM Evaluation ---
                   precision    recall  f1-score   support

      Factual (0)       0.45      0.47      0.46       984
Hallucination (1)       0.47      0.45      0.46      1016

         accuracy                           0.46      2000
        macro avg       0.46      0.46      0.46      2000
     weighted avg       0.46      0.46      0.46      2000

AUROC: 0.5534
AUPRC: 0.5336

--- XGBoost Evaluation ---
                   precision    recall  f1-score   support

      Factual (0)       0.47      0.47      0.47       984
Hallucination (1)       0.49      0.49      0.49      1016

         accuracy                           0.48      2000
        macro avg       0.48      0.48      0.48      2000
     weighted avg       0.48      0.48      0.48      2000

AUROC: 0.4635
AUPRC: 0.4930
